In [47]:
import pandas as pd
import numpy as np
import glob
from tqdm import tqdm
import re
import json

def printShape(df, cols=[], msg=''):
    
    print(df.shape, end='  ')
    for col in cols:
        if col in df.columns:
            print(col, df[col].nunique(), end='  ')
    print(msg, flush=True)
    
    return df

PROJDATA = '../data'
PLOS = f'{PROJDATA}/plos'
LARGE = f'{PROJDATA}/large_files'
EXTERNAL = '/scratch/fl1092/LLM_use_RAD/data/PLOS_metadata'
EXTERNAL2 = '/scratch/fl1092/LLM_use_RAD/data/LLM_detect_plosreveiw'

# Paper country

In [6]:
def processNorthSouth(row):

    # True: global south
    # False: global north

    r = row['region']
    sub = row['sub-region']
    country = row['country']

    if country == 'JP' or country == 'IL' or country == "KR":
        # Japan, Israel, and Korea are global north
        return False

    if r == 'Asia' or r == 'Africa':
        return True
    elif r == 'Americas':
        if sub == 'Latin America and the Caribbean':
            return True
        elif sub == 'Northern America':
            return False
        else:
            raise Error('Unknown subregion in Americas')
    elif r == 'Oceania':
        if sub == 'Australia and New Zealand':
            return False
        else:
            return True
    elif r == "Europe":
        return False

continents = (
    pd.read_csv(
        f'{PROJDATA}/continents2.csv',
        usecols=['alpha-2', 'region', 'sub-region', 'name'],
        keep_default_na=False
    )
    .rename(columns={'alpha-2': 'country'})
    .pipe(printShape)
    
    .assign(GlobalSouth=lambda df: df.apply(processNorthSouth, axis=1))
    .assign(name=lambda df: df.name.apply(lambda x: x.lower()))
)

(249, 4)  


In [7]:
%%time
import numpy as np

def parseCountry(x):
    
    m = {
        'united states of america': 'united states',
        'usa': 'united states',

        'united kingdom of great britain and northern ireland': 'united kingdom',
        'england': 'united kingdom',
        
        'the netherlands':'netherlands',
        
        'korea':'korea, republic of',
        'republic of korea':'korea, republic of',
        
        'people’s republic of china':'china',
        "people's republic of china":'china',
        'p. r. china':'china',
        'p.r. china':'china',
        'pr china':'china',
        'republic of china':'china',
        'roc':'china',
        
        'méxico':'mexico',
        'brasil':'brazil',
        'islamic republic of iran':'iran',
        'russian federation':'russia',
        'türkiye':'turkey',
        'viet nam':'vietnam',
        'kingdom of saudi arabia':'saudi arabia',
        'perú':'peru'
    }
    
    try:
        x = x.split(',')[-1].strip().lower()
        
        if x in m:
            return m[x]
        return x
    except Exception as e:
        return np.nan

papAff = (
    pd.read_csv(f'{EXTERNAL}/papers_affiliations.csv').pipe(printShape, cols=['paper_id'])
    .assign(Country=lambda df: df.affiliation_text.apply(parseCountry))
)# (1778561, 3)  paper_id 369209

(1778561, 3)  paper_id 369209  
CPU times: user 7.86 s, sys: 279 ms, total: 8.14 s
Wall time: 8.17 s


In [8]:
papAuAff = (
    pd.read_csv(f'{EXTERNAL}/papers_author_affiliations.csv')
    .pipe(printShape, cols=['paper_id'])
    
    # remove editor affiliations
    .query('affiliation_id != "edit1"')
    .pipe(printShape, cols=['paper_id'])
) # (3272522, 3)  paper_id 387157 

(3272522, 3)  paper_id 387157  
(3272522, 3)  paper_id 387157  


In [9]:
papAuthorCount = (
    papAuAff.groupby('paper_id').author_full_name.nunique().reset_index()
    .rename(columns={'author_full_name':'AuthorCount'})
    
    .pipe(printShape, cols=['paper_id'])
) # 387157, 2  paper_id 387157 

(387157, 2)  paper_id 387157  


In [10]:
papAuAffIDCount = (
    papAuAff.groupby(['paper_id','affiliation_id']).author_full_name.nunique().reset_index()
    .rename(columns={'author_full_name':'AuthorCount'})
    
    .pipe(printShape, cols=['paper_id'])
    
    .merge(papAuthorCount.rename(columns={'AuthorCount':'Total'}))
    .assign(Percentage=lambda df: df.AuthorCount / df.Total)
    .pipe(printShape, cols=['paper_id'])
) # (1412531, 3)  paper_id 369050  

(1412531, 3)  paper_id 369050  
(1412531, 5)  paper_id 369050  


In [11]:
papCountry = (
    papAuAffIDCount.merge(papAff, on=['paper_id','affiliation_id'])
    .pipe(printShape, cols=['paper_id'])
    
    .sort_values(by='Percentage',ascending=False)
    .groupby(['paper_id','Country']).head(1)
    
    .pipe(printShape, cols=['paper_id'])
)

papMainCountry = (
    papCountry.drop_duplicates(subset=['paper_id'], keep='first')
    .pipe(printShape, cols=['PLOSPaperID'])
    
    .assign(PLOSPaperID=lambda df: df.paper_id.apply(lambda x: x.split('/')[1].replace('journal.','')))
    
    .drop_duplicates(subset=['PLOSPaperID'], keep='first')
    [['PLOSPaperID', 'Country', 'Total']]
    
    .merge(continents, left_on='Country', right_on='name')
    .pipe(printShape, cols=['PLOSPaperID'])
)
# (1412531, 7)  paper_id 369050  
# (579278, 7)  paper_id 369050  
# (369050, 7)  paper_id 369050  
# 363813

(1412531, 7)  paper_id 369050  
(577279, 7)  paper_id 369050  
(369050, 7)  
(363813, 8)  PLOSPaperID 363813  


In [12]:
papMainCountry.to_csv(f'{PLOS}/PaperMainCountry.csv', index=False)

# Reviewer comments

In [13]:
import numpy as np

def extractWritingQuality(x):
    
    x = x.lower().replace(u'\xa0', u' ')
    
    if '**********' in x:
        parts = x.split('**********')
    
    elif '--------------------' in x:
        parts = x.split('--------------------')
        
    elif '________________________________________' in x:
        parts = x.split('________________________________________')
        
    else:
        parts = []
    
    for part in parts:
        if 'is the manuscript presented in an intelligible fashion and written in standard english' in part:
            return part.split('\n')
        
    return np.nan

In [14]:
import numpy as np

def extractPaperQuality(x):
    
    x = x.lower().replace(u'\xa0', u' ')
    
    if '**********' in x:
        parts = x.split('**********')
    
    elif '--------------------' in x:
        parts = x.split('--------------------')
        
    elif '________________________________________' in x:
        parts = x.split('________________________________________')
        
    else:
        parts = []
    
    for part in parts:
        if 'is the manuscript technically sound, and do the data support the conclusions' in part:
            return part.split('\n')
        
    return np.nan

In [15]:
import numpy as np

def extractFreeStyleComments(x):
    
    x = x.lower().replace(u'\xa0', u' ')
    
    if '**********' in x:
        parts = x.split('**********')
    
    elif '--------------------' in x:
        parts = x.split('--------------------')
        
    elif '________________________________________' in x:
        parts = x.split('________________________________________')
        
    else:
        parts = []
    
    for part in parts:
        if 'review comments to the author' in part:
            return part.split('\n')
        
    return np.nan

In [16]:
reviewDates = (
    pd.read_csv(f'{PLOS}/peer_review_dates.csv', parse_dates=['date'])
    .pipe(printShape, cols=['sub_article_id'])
)

(219947, 2)  sub_article_id 219947  


In [17]:
%%time
reviewContent = (
    pd.read_csv(
        f'{EXTERNAL}/peer_review_history.csv', usecols=['sub_article_id','content','article_title']
    ).pipe(printShape, cols=['sub_article_id'])

    ### Keep editor comments only ###
    .assign(IsAuthorResponse=lambda df: df.article_title.apply(lambda x: 'author response to' in x.lower()))
    .query('IsAuthorResponse==False').drop('IsAuthorResponse', axis=1)
    .pipe(printShape, cols=['sub_article_id'])
    
    .assign(PLOSPaperID=lambda df: df.sub_article_id.apply(lambda x: '.'.join(x.split('.')[:2])))
    .assign(Journal=lambda df: df.sub_article_id.apply(lambda x: x.split('.')[0]))
    .dropna()
    .pipe(printShape, cols=['sub_article_id','PLOSPaperID'], msg='drop na')
    
    .assign(PaperQualityComment=lambda df: df.content.apply(extractPaperQuality))
    .assign(LanguageQualityComment=lambda df: df.content.apply(extractWritingQuality))
    
    .merge(reviewDates, on='sub_article_id')
    .pipe(printShape, cols=['sub_article_id','PLOSPaperID'], msg='get review dates')
)

# (225423, 3)  sub_article_id 225423  
# (153912, 3)  sub_article_id 153912  
# (153912, 5)  sub_article_id 153912  PLOSPaperID 44999  
# (149690, 8)  sub_article_id 149690  PLOSPaperID 44631  

(225423, 3)  sub_article_id 225423  
(153912, 3)  sub_article_id 153912  
(153912, 5)  sub_article_id 153912  PLOSPaperID 44999  drop na
(149690, 8)  sub_article_id 149690  PLOSPaperID 44631  get review dates
CPU times: user 44.7 s, sys: 1.31 s, total: 46 s
Wall time: 46.1 s


In [18]:
reviewContent.article_title.value_counts().head(3)
# Decision Letter 0     44628
# Decision Letter 1     43998
# Acceptance letter     37110

article_title
Decision Letter 0    44628
Decision Letter 1    43998
Acceptance letter    37110
Name: count, dtype: int64

In [21]:
firstRound=(
    reviewContent.sort_values(by='date')
    .drop_duplicates(subset=['PLOSPaperID'], keep='first')
    .assign(FirstRound=True) # 44631
    
    .pipe(printShape, cols=['sub_article_id','PLOSPaperID'], msg='first round')
    [['sub_article_id','FirstRound']]
)

lastRound=(
    reviewContent.sort_values(by='date')
    .dropna(subset=['LanguageQualityComment'])
    .drop_duplicates(subset=['PLOSPaperID'], keep='last')
    .assign(LastRound=True) # 44631
    
    .pipe(printShape, cols=['sub_article_id','PLOSPaperID'], msg='last round')
    [['sub_article_id','LastRound']]
)
# (44631, 9)  sub_article_id 44631  PLOSPaperID 44631  first round
# (36201, 9)  sub_article_id 36201  PLOSPaperID 36201  last round

(44631, 9)  sub_article_id 44631  PLOSPaperID 44631  first round
(36201, 9)  sub_article_id 36201  PLOSPaperID 36201  last round


In [22]:
%%time
firstRound.to_csv(f'{PLOS}/PeerReviewFirstRounds.csv', index=False)
lastRound.to_csv(f'{PLOS}/PeerReviewLastRounds.csv', index=False)

CPU times: user 91 ms, sys: 2.12 ms, total: 93.1 ms
Wall time: 96.4 ms


In [23]:
import re

def extractReviewerScore(lines):
    """
    Given a list of strings, extract reviewer id and answer.

    Returns:
        List of tuples: [(reviewer_id, answer), ...]
    """

    for line in lines:
        if not line:
            continue

        m = re.search(r"reviewer\s*#(\d+)\s*:\s*(.+)", line, re.IGNORECASE)
        if m:
            reviewer_id = int(m.group(1))
            answer = m.group(2).strip()
            yield reviewer_id, answer
            
            
languageRating = []

for ind, row in reviewContent.iterrows():
    
    if type(row['LanguageQualityComment']) is float:
        continue
    
    for reviewerID, answer in extractReviewerScore(row['LanguageQualityComment']):
        
        languageRating.append({
            'PLOSPaperID': row['PLOSPaperID'],
            'article_title': row['article_title'],
            'sub_article_id': row['sub_article_id'],
            'date': row['date'],
            'Reviewer': reviewerID,
            'Rating': answer,
        })
        
languageRating = (
    pd.DataFrame(languageRating)
    .pipe(printShape, cols=['sub_article_id','PLOSPaperID']) # (136747, 6)  sub_article_id 69687  PLOSPaperID 36199 
    
    .drop_duplicates()
    .pipe(printShape, cols=['sub_article_id','PLOSPaperID']) # (136711, 6)  sub_article_id 69687  PLOSPaperID 36199  
)

languageRating = (
    languageRating[languageRating.Rating.isin(['yes', 'no', '(no response)', 'partly', "i don't know"])]
    .pipe(printShape, cols=['sub_article_id','PLOSPaperID']) # (136677, 6)  sub_article_id 69685  PLOSPaperID 36198  
    
    .drop_duplicates()
    .pipe(printShape, cols=['sub_article_id','PLOSPaperID']) # (136677, 6)  sub_article_id 69685  PLOSPaperID 36198  
    
    .drop_duplicates(subset=['sub_article_id', 'Reviewer'], keep=False)
    .pipe(printShape, cols=['sub_article_id','PLOSPaperID']) # (136647, 6)  sub_article_id 69681  PLOSPaperID 36197  
)

(136747, 6)  sub_article_id 69687  PLOSPaperID 36199  
(136711, 6)  sub_article_id 69687  PLOSPaperID 36199  
(136677, 6)  sub_article_id 69685  PLOSPaperID 36198  
(136677, 6)  sub_article_id 69685  PLOSPaperID 36198  
(136647, 6)  sub_article_id 69681  PLOSPaperID 36197  


In [24]:
paperRating = []

for ind, row in reviewContent.iterrows():
    
    if type(row['PaperQualityComment']) is float:
        continue
    
    for reviewerID, answer in extractReviewerScore(row['PaperQualityComment']):
        
        paperRating.append({
            'PLOSPaperID': row['PLOSPaperID'],
            'article_title': row['article_title'],
            'sub_article_id': row['sub_article_id'],
            'date': row['date'],
            'Reviewer': reviewerID,
            'Rating': answer,
        })
        
paperRating = (
    pd.DataFrame(paperRating)
    .pipe(printShape, cols=['sub_article_id','PLOSPaperID']) # (133081, 6)  sub_article_id 67812  PLOSPaperID 35199  
    
    .drop_duplicates()
    .pipe(printShape, cols=['sub_article_id','PLOSPaperID']) # (133055, 6)  sub_article_id 67812  PLOSPaperID 35199  
)

paperRating = (
    paperRating[paperRating.Rating.isin(['yes', 'no', '(no response)', 'partly', "i don't know"])]
    .pipe(printShape, cols=['sub_article_id','PLOSPaperID']) # (132986, 6)  sub_article_id 67808  PLOSPaperID 35197  
    
    .drop_duplicates()
    .pipe(printShape, cols=['sub_article_id','PLOSPaperID']) # (132986, 6)  sub_article_id 67808  PLOSPaperID 35197  
    
    .drop_duplicates(subset=['sub_article_id', 'Reviewer'], keep=False)
    .pipe(printShape, cols=['sub_article_id','PLOSPaperID']) # (132957, 6)  sub_article_id 67804  PLOSPaperID 35197 
)

(133081, 6)  sub_article_id 67812  PLOSPaperID 35199  
(133055, 6)  sub_article_id 67812  PLOSPaperID 35199  
(132986, 6)  sub_article_id 67808  PLOSPaperID 35197  
(132986, 6)  sub_article_id 67808  PLOSPaperID 35197  
(132957, 6)  sub_article_id 67804  PLOSPaperID 35197  


In [27]:
%%time
languageRating.to_csv(f'{PLOS}/LanguageRating.csv', index=False)
paperRating.to_csv(f'{PLOS}/PaperRating.csv', index=False)

CPU times: user 941 ms, sys: 13 ms, total: 954 ms
Wall time: 959 ms


## LLM-assistance of reviewer comments

In [55]:
reviewLLM = (
    pd.read_csv(
        f'{EXTERNAL2}/PeerReviewClassified.csv',
        usecols=['DOI', 'RoundID', 'ReviewerID', 'Score', 'CI', 'Len']
    )
    
    .query('Score != "ERROR"')
    .assign(Score=lambda df: df.Score.astype(float))
    .assign(ReviewerLLM=lambda df: df.Score >= 0.5)
    
    .merge(
        (
            reviewDates
            .rename(columns={'date':'curr_date', 'sub_article_id':'RoundID'})
            .assign(PLOSPaperID=lambda df: df.RoundID.apply(lambda x: '.'.join(x.split('.')[:-1])))
        ), on='RoundID'
    )
   
    .groupby(['PLOSPaperID','curr_date','RoundID']).ReviewerLLM.any().reset_index()
    .pipe(printShape, cols=['RoundID']) # 18503
)

(18503, 4)  RoundID 18503  


In [56]:
reviewLLM.to_csv(f'{PLOS}/ReviewerUsingLLM.csv', index=False)

# Peer review turnaround

In [32]:
def extractRole(x):
    x = x.lower()
    
    if "author response to decision letter" in x:
        return 'author'
    
    if "decision letter" in x or "acceptance letter" in x:
        return 'editor'
    
    return 'UNKNOWN' # `Author response to previous submission`

In [35]:
def parseDate(date):
    
    parsed = pd.to_datetime(date, errors='coerce', format='%d %b %Y')
    if not pd.isna(parsed):
        return parsed
    
    parsed = pd.to_datetime(date, errors='coerce', format='%b %d %Y')
    if not pd.isna(parsed):
        return parsed
    
    parsed = pd.to_datetime(date, errors='coerce', format='%B %d %Y')
    if not pd.isna(parsed):
        return parsed
    
    parsed = pd.to_datetime(date, errors='coerce', format='%d %B %Y')
    if not pd.isna(parsed):
        return parsed

In [36]:
%%time
reviewDf = (
    pd.read_csv(
        f'{LARGE}/peer_review_history.csv', parse_dates=['date'],
        usecols=['sub_article_id','article_title','date']
    )
    
    .assign(date=lambda df: df.date.apply(parseDate))
    .dropna()
    
    .assign(PLOSPaperID=lambda df: df.sub_article_id.apply(lambda x: '.'.join(x.split('.')[:2])))
    .assign(Journal=lambda df: df.sub_article_id.apply(lambda x: x.split('.')[0]))
    .assign(Role=lambda df: df.article_title.apply(extractRole))
    
    .pipe(printShape, cols=['PLOSPaperID'])
    
    .query('Role != "UNKNOWN"')
    .pipe(printShape, cols=['PLOSPaperID'])
)
# (221190, 6)  PLOSPaperID 44998  
# (219947, 6)  PLOSPaperID 44998

(221190, 6)  PLOSPaperID 44998  
(219947, 6)  PLOSPaperID 44998  
CPU times: user 36.9 s, sys: 407 ms, total: 37.3 s
Wall time: 37.5 s


## Turnaround of each round of review

In [37]:
import pandas as pd

def fix_same_day_initial_response(df):
    """
    Cleaning step:
    For each PLOSPaperID, if there is an 'Author response to Decision Letter 0'
    on the same day as 'Original Submission', remove that row and
    decrement the round number of all subsequent author responses by 1.
    """
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])

    def _fix_one_paper(g: pd.DataFrame) -> pd.DataFrame:
        g = g.copy()

        # find the Original Submission date (earliest if multiple)
        sub_mask = g["article_title"] == "Original Submission"
        if not sub_mask.any():
            return g

        sub_date = g.loc[sub_mask, "date"].min()

        # find Author response to Decision Letter 0 on the same date
        resp0_mask = (
            (g["article_title"] == "Author response to Decision Letter 0") &
            (g["date"] == sub_date)
        )

        # if none, we don't change anything for this paper
        if not resp0_mask.any():
            return g

        # drop those same-day response-0 rows
        g = g.loc[~resp0_mask].copy()

        # now shift all later author responses (n >= 1) down by 1
        resp_mask = g["article_title"].str.startswith("Author response to Decision Letter")

        # extract the numeric round at the end of the string
        round_num = g.loc[resp_mask, "article_title"].str.extract(r"(\d+)$")[0].astype(int)

        # only shift n >= 1
        shift_mask = resp_mask.copy()
        shift_mask.loc[resp_mask] &= round_num >= 1

        if shift_mask.any():
            new_rounds = (round_num[round_num >= 1] - 1).astype(str)
            g.loc[shift_mask, "article_title"] = (
                "Author response to Decision Letter " + new_rounds
            )

        return g

    # apply per paper
    out = (
        df.groupby("PLOSPaperID", group_keys=False)
          .apply(_fix_one_paper)
          .reset_index(drop=True)
    )
    return out

In [39]:
%%time
subDates = (
    pd.read_csv(
        f"{PLOS}/papers_acceptance_delay.csv", parse_dates=['submission_date'],
        usecols=['paper_id','submission_date']
    )
    
    .assign(PublisherCode=lambda df: df.paper_id.apply(lambda x: x.split('/')[0]))
    .assign(PLOSPaperID=lambda df: df.paper_id.apply(lambda x: x.split('/')[1].replace('journal.','')))
    .drop('paper_id', axis=1)
    
    .merge(reviewDf[['PLOSPaperID']].drop_duplicates(), on='PLOSPaperID')
    .pipe(printShape, cols=['PLOSPaperID']) # 44998
)

(361676, 2)  paper_id 361676  
(361676, 3)  PLOSPaperID 361676  
(44998, 3)  PLOSPaperID 44998  
CPU times: user 1.14 s, sys: 24.9 ms, total: 1.16 s
Wall time: 1.18 s


In [40]:
%%time
completeReviewDf = (
    pd.concat([(
        subDates[['PLOSPaperID','submission_date']]
        .rename(columns={'submission_date':'date'})
        .assign(Role='author')
        .assign(article_title='Original Submission')
    ), reviewDf.drop(['Journal','sub_article_id'], axis=1)], ignore_index=True, sort=False)
    .pipe(printShape, cols=['PLOSPaperID'])
    
    # there are cases where the first submission is immediate followed by an "author response to letter 0"
    # remove the first occurance of such cases
    .sort_values(by='date')
    .drop_duplicates(subset=['PLOSPaperID', 'article_title'], keep='last')
    .pipe(printShape, cols=['PLOSPaperID'])
    
    .pipe(fix_same_day_initial_response)
    .pipe(printShape, cols=['PLOSPaperID'])
)
# (264945, 4)  PLOSPaperID 44998  
# (264249, 4)  PLOSPaperID 44998  
# (261386, 4)  PLOSPaperID 44998 

(264945, 4)  PLOSPaperID 44998  
(264249, 4)  PLOSPaperID 44998  
(261386, 4)  PLOSPaperID 44998  
CPU times: user 46.3 s, sys: 400 ms, total: 46.7 s
Wall time: 46.9 s


/tmpdata/ipykernel_2130129/2212565893.py:57: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_fix_one_paper)


## Distinguibh between editor duration and author duration

In [41]:
import pandas as pd

def _add_round_column(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add a 'round' column for decision/response events.
    'Decision Letter n' -> round = n
    'Author response to Decision Letter n' -> round = n
    Other rows get NA (will be handled separately if needed).
    """
    df = df.copy()
    # extract trailing integer if present
    r = df["article_title"].str.extract(r"(\d+)$")[0]
    df["round"] = r.astype("float").astype("Int64")  # nullable int
    return df

In [42]:
def author_revision_times(df: pd.DataFrame) -> pd.DataFrame:
    """
    Author time: from Decision Letter n (editor) to
    Author response to Decision Letter n (author).
    """
    df = _add_round_column(df)
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])

    decisions = (
        df[df["article_title"].str.startswith("Decision Letter")
           & (df["Role"].str.lower() == "editor")]
        [["PLOSPaperID", "round", "date"]]
        .rename(columns={"date": "decision_date"})
    )

    responses = (
        df[df["article_title"].str.startswith("Author response to Decision Letter")
           & (df["Role"].str.lower() == "author")]
        [["PLOSPaperID", "round", "date"]]
        .rename(columns={"date": "response_date"})
    )

    out = decisions.merge(
        responses,
        on=["PLOSPaperID", "round"],
        how="inner"
    )

    out["duration_days"] = (out["response_date"] - out["decision_date"]).dt.days
    out["stage_type"] = "author_revision"  # optional label

    return out

In [43]:
def editor_processing_times(df: pd.DataFrame) -> pd.DataFrame:
    """
    Editor time:

    - From Original Submission (author) → Decision Letter 0 (editor)
    - From Author response to Decision Letter n (author) → Decision Letter (n+1) (editor)
    """
    df = _add_round_column(df)
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])

    # ---- Stage 0: Original Submission → Decision Letter 0 ----
    submissions = (
        df[(df["article_title"] == "Original Submission")
           & (df["Role"].str.lower() == "author")]
        [["PLOSPaperID", "date"]]
        .rename(columns={"date": "prev_date"})
    )

    decision0 = (
        df[(df["article_title"] == "Decision Letter 0")
           & (df["Role"].str.lower() == "editor")]
        [["PLOSPaperID", "date"]]
        .rename(columns={"date": "curr_date"})
    )

    stage0 = submissions.merge(decision0, on="PLOSPaperID", how="inner")
    if not stage0.empty:
        stage0["editor_stage"] = 0
        stage0["prev_event"] = "Original Submission"
        stage0["curr_event"] = "Decision Letter 0"
        stage0["duration_days"] = (stage0["curr_date"] - stage0["prev_date"]).dt.days

    # ---- Later stages: Author response n → Decision Letter (n+1) ----
    responses = (
        df[df["article_title"].str.startswith("Author response to Decision Letter")
           & (df["Role"].str.lower() == "author")]
        [["PLOSPaperID", "round", "date"]]
        .rename(columns={"date": "prev_date"})
    )

    decisions = (
        df[df["article_title"].str.startswith("Decision Letter")
           & (df["Role"].str.lower() == "editor")]
        [["PLOSPaperID", "round", "article_title", "date"]]
        .rename(columns={"date": "curr_date"})
    )

    # editor stage index is the *decision* round:
    # stage 1: response to DL0 -> DL1, etc.
    decisions["editor_stage"] = decisions["round"]

    # align response_n with decision_(n+1)
    dec_shift = decisions.copy()
    dec_shift["round"] = dec_shift["round"] - 1  # key for merge

    later = responses.merge(
        dec_shift,
        on=["PLOSPaperID", "round"],
        how="inner",
        suffixes=("", "_dec")
    )

    if not later.empty:
        later["prev_event"] = "Author response to Decision Letter " + later["round"].astype(str)
        later["curr_event"] = later["article_title"]
        later["duration_days"] = (later["curr_date"] - later["prev_date"]).dt.days

    # ---- Combine all editor stages ----
    all_editor = []
    if not stage0.empty:
        all_editor.append(stage0[[
            "PLOSPaperID", "editor_stage",
            "prev_event", "curr_event",
            "prev_date", "curr_date", "duration_days"
        ]])
    if not later.empty:
        all_editor.append(later[[
            "PLOSPaperID", "editor_stage",
            "prev_event", "curr_event",
            "prev_date", "curr_date", "duration_days"
        ]])

    if not all_editor:
        return pd.DataFrame(
            columns=[
                "PLOSPaperID", "editor_stage",
                "prev_event", "curr_event",
                "prev_date", "curr_date", "duration_days"
            ]
        )

    out = pd.concat(all_editor, ignore_index=True).sort_values(
        ["PLOSPaperID", "editor_stage"]
    )

    return out

In [44]:
%%time
authorDuration = (
    author_revision_times(completeReviewDf)
    .pipe(printShape, cols=['PLOSPaperID'])
    
    .query('duration_days >= 0')
    .pipe(printShape, cols=['PLOSPaperID'])
)
# (66136, 6)  PLOSPaperID 43933  
# (66050, 6)  PLOSPaperID 43880

(66136, 6)  PLOSPaperID 43933  
(66050, 6)  PLOSPaperID 43880  
CPU times: user 815 ms, sys: 2.98 ms, total: 817 ms
Wall time: 821 ms


In [45]:
%%time
editorDuration = (
    editor_processing_times(completeReviewDf)
    .pipe(printShape, cols=['PLOSPaperID'])
    
    .query('duration_days >= 0')
    .pipe(printShape, cols=['PLOSPaperID'])
)
# (110701, 7)  PLOSPaperID 44631  
# (110690, 7)  PLOSPaperID 44631  

(110701, 7)  PLOSPaperID 44631  
(110690, 7)  PLOSPaperID 44631  
CPU times: user 1.21 s, sys: 4.97 ms, total: 1.22 s
Wall time: 1.22 s


In [46]:
editorDuration.to_csv(f'{PLOS}/PeerReviewEditorDuration.csv', index=False)
authorDuration.to_csv(f'{PLOS}/PeerReviewAuthorDuration.csv', index=False)